In [1]:
from OceanDataStore import OceanDataCatalog
from nemo_cookbook import NEMODataTree
import gsw
import xarray as xr
import os 
import dask 
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
import pandas as pd

from helper_function import solve_vertical_modes
from scipy.linalg import eigh

/home/teaching/StreamfunctionReconstruction/venv/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=1,
    memory_limit="8GB"
)

client = Client(cluster)

print(client.dashboard_link)

http://127.0.0.1:8787/status


In [4]:
catalog = OceanDataCatalog(catalog_name="noc-stac") 

ds_domain = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/domain_cfg", consolidated=True, chunks={})

ds_gridT = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/T1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})
ds_gridU = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/U1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})
ds_gridV = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/V1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})

ds_gridT_y = ds_gridT.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "deptht": 25, "y": 100, "x": 120})
ds_gridT_y["sigma0"] = gsw.density.sigma0(CT=ds_gridT_y["thetao_con"], SA=ds_gridT_y["so_abs"])
ds_gridT_y["sigma0"].name = "sigma0"
ds_gridU_y = ds_gridU.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "depthu": 25, "y": 100, "x": 120})
ds_gridV_y = ds_gridV.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "depthv": 25, "y": 100, "x": 120})

datasets_y = {"parent": {"domain": ds_domain, "gridT": ds_gridT_y, "gridU": ds_gridU_y, "gridV": ds_gridV_y}}

# Initialise a new NEMODataTree whose parent domain is zonally periodic & north-folding on F-points:
nemo_y = NEMODataTree.from_datasets(datasets=datasets_y, iperio=True, nftype="F", read_mask=True)

###
# overturnign streamfunction #
###
atlmask = ds_domain['atlmsk'].rename({"x":"i", "y":"j"}).astype(bool) # assign to V-grid
atlmask['i'] = atlmask['i'] + 1
atlmask['j'] = atlmask['j'] + 1.5
Psi = 1E-6 * nemo_y['gridV/vo'].integral(dims=["i", "k"], cum_dims=["k"], dir="+1", mask=atlmask)
print('Overturning streamfunciton calculated')

Overturning streamfunciton calculated


In [3]:
catalog = OceanDataCatalog(catalog_name="noc-stac") 

ds_domain = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/domain_cfg", consolidated=True, chunks={})

ds_gridT = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/T1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})
ds_gridU = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/U1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})
ds_gridV = xr.open_zarr("https://noc-msm-o.s3-ext.jc.rl.ac.uk/npd-eorca1-jra55v1/V1m", consolidated=True, chunks={"time_counter": 4, "k": 25, "j": 100, "i": 120})

ds_gridT_y = ds_gridT.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "deptht": 25, "y": 100, "x": 120})
ds_gridT_y["sigma0"] = gsw.density.sigma0(CT=ds_gridT_y["thetao_con"], SA=ds_gridT_y["so_abs"])
ds_gridT_y["sigma0"].name = "sigma0"
ds_gridU_y = ds_gridU.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "depthu": 25, "y": 100, "x": 120})
ds_gridV_y = ds_gridV.resample(time_counter="1YE").mean().chunk({"time_counter": 5, "depthv": 25, "y": 100, "x": 120})

datasets_y = {"parent": {"domain": ds_domain, "gridT": ds_gridT_y, "gridU": ds_gridU_y, "gridV": ds_gridV_y}}

# Initialise a new NEMODataTree whose parent domain is zonally periodic & north-folding on F-points:
nemo_y = NEMODataTree.from_datasets(datasets=datasets_y, iperio=True, nftype="F", read_mask=True)

###
# overturnign streamfunction #
###
atlmask = ds_domain['atlmsk'].rename({"x":"i", "y":"j"}).astype(bool) # assign to V-grid
atlmask['i'] = atlmask['i'] + 1
atlmask['j'] = atlmask['j'] + 1.5
Psi = nemo_y['gridV/vo'].integral(dims=["i", "k"], cum_dims=["k"], dir="+1", mask=atlmask)
print('Overturning streamfunciton calculated')

Overturning streamfunciton calculated


In [5]:
j_index = np.arange(225, 255)

Psi_subset = Psi.isel(j=j_index)

task = Psi_subset.to_netcdf("Psij_scaled.nc", compute=False)

print("Number of tasks:", len(task.dask))

future = client.compute(task)

Number of tasks: 6433


In [ ]:
# g = 9.8 # gravity 
# rho_0 = 1026 # reference sea water denisty

# ## CALCULATE N2 ACROSS OCEAN ##
# atlmask_T = (ds_domain["atlmsk"].rename({"x": "i", "y": "j"}).astype(bool))
# atlmask_T = atlmask_T.assign_coords(i=nemo_y["gridT/sigma0"].i,j=nemo_y["gridT/sigma0"].j)

# sigma0_mean = (nemo_y["gridT/sigma0"].where(atlmask_T).mean( dim=["time_counter", "j", "i"], skipna=True))

# N2_mean = (g / rho_0 * sigma0_mean.differentiate("deptht")) # originally ran with negative value
# N2_mean.name = "N2" # rename variable calculated as N2 instead of signam0
# print("Mean N2 calculated")

# with ProgressBar():
#     N2_mean.to_netcdf("N2_mean.nc", compute=True)